<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/11_GES_Aware_Genomic_RAG_Cell_7C4_Score_Blind_Prompt_Materialization_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm that Google Drive is mounted and the project folder name is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, frozen identities, exact checksums, and fail-closed output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import csv
import hashlib
import json
import re
import shutil
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = '11_GES_Aware_Genomic_RAG_Cell_7C4_Score_Blind_Prompt_Materialization_V3.ipynb'
CELL_ID = '7C4'
STAGE = '7C'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_QUESTIONS = 80
EXPECTED_ALIASES = 6
EXPECTED_CONTEXTS_PER_PROMPT = 5
EXPECTED_PROMPTS = 480
EXPECTED_CONTEXT_ROWS = 2_400
EXPECTED_FUTURE_REPETITIONS = 3
EXPECTED_FUTURE_CALLS = 1_440

EXPECTED_CELL_7C3_AUTHORIZATION_DECISION = (
    'AUTHORIZE_STAGE7C_CELL7C4_SCORE_BLIND_PROMPT_MATERIALIZATION_ONLY_'
    '480_QUESTION_ALIAS_PROMPTS_FIVE_FROZEN_CONTEXT_BLOCKS_EACH_EXACT_CELL7B4_TEMPLATES_'
    'NO_SCORE_BEARING_AUDIT_GES_QUALITY_RANK_RRF_CONDITION_IDENTITY_ANSWER_KEYS_LLM_OR_METRICS'
)

EXPECTED_CELL_7C3_TERMINAL_DECISION = (
    'PASS_STAGE7C3_COMPLETE_CELL7C2_SCORE_BLIND_TOP5_CELL7B3_SEMANTIC_CORPUS_AND_QUESTIONS_'
    'AND_CELL7B4_PROMPT_LLM_CONFIGURATION_REVERIFIED_CHECKSUM_PROTECTED_CELL7C4_'
    '480_SCORE_BLIND_PROMPT_MATERIALIZATION_ONLY_AUTHORIZED_NO_SCORE_BEARING_AUDIT_GES_'
    'QUALITY_RANK_RRF_CONDITION_IDENTITY_ANSWER_KEYS_LLM_RESPONSES_ADJUDICATION_OR_RAG_METRICS'
)

EXPECTED_CELL_7C2_TERMINAL_DECISION = (
    'PASS_STAGE7C2_FROZEN_CELL7A3_SCORES_JOINED_TO_EXACT_CELL7C0_TOP20_'
    'SIX_CONDITION_QUALITY_RANKS_FIXED_075_SEMANTIC_025_QUALITY_RRF_CONSTANT60_'
    'AND_2400_SCORE_BLIND_FINAL_TOP5_SELECTIONS_MATERIALIZED_CHECKSUM_PROTECTED_'
    'NO_NEW_RETRIEVAL_HARD_EXCLUSION_PROMPTS_LLM_ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS_'
    'NEXT_EXECUTION_NOT_AUTHORIZED'
)

EXPECTED_CELL_7B4_TERMINAL_DECISION = (
    'PASS_STAGE7B4_EXACT_EMBEDDING_MODEL_REVISION_TEXT_NORMALIZATION_'
    'SIMILARITY_TOP20_CANDIDATE_POOL_TOP5_CONTEXT_QUALITY_RERANKING_'
    'BLINDED_ALIASES_PROMPTS_STRICT_RESPONSE_SCHEMA_FIXED_LLM_SNAPSHOT_'
    'GENERATION_RUNTIME_AND_DETERMINISTIC_CONTROLS_FROZEN_CHECKSUM_PROTECTED_'
    'NO_EMBEDDINGS_RETRIEVAL_RERANKING_PROMPT_MATERIALIZATION_LLM_'
    'ANSWER_KEY_OUTCOME_INSPECTION_OR_RAG_EVALUATION_EXECUTION_NOT_AUTHORIZED'
)

# --------------------------------------------------------------------------------------------------
# Successful Cell 7C3 V3 package — outputs were intentionally written to the v2 package directory.
# --------------------------------------------------------------------------------------------------
CELL_7C3_AUTH_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c3_prompt_materialization_authorization_v2'
)
CELL_7C3_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c3_prompt_materialization_authorization_v2'
)

CELL_7C3 = OrderedDict([
    ('authorization', {
        'path': CELL_7C3_AUTH_DIR / 'cell_7c3_stage7c_cell7c4_prompt_materialization_authorization_v2.json',
        'sha256': '6872aa3fe7349d116985ef32e3d99e609234a1be3f506fd7dea317d9a8328a71',
    }),
    ('input_inventory', {
        'path': CELL_7C3_AUTH_DIR / 'cell_7c3_authorized_prompt_materialization_input_inventory_v2.csv',
        'sha256': 'da80081c1ce5b1ca8fa484399e9a0f6d307a67bc81b1fad9d367e9e614838ff5',
    }),
    ('qc', {
        'path': CELL_7C3_QC_DIR / 'cell_7c3_prompt_materialization_authorization_qc_v2.json',
        'sha256': 'c8452c8c3432ce2f7d84051f29e1ebb0a52ef820d7464fa14be61b62dcbed644',
    }),
    ('manifest', {
        'path': CELL_7C3_AUTH_DIR / 'cell_7c3_prompt_materialization_authorization_manifest_v2.json',
        'sha256': 'fc2ddb9593baf025e688b12254757fa9125e667ea57ad19104b2179cb5243f0c',
    }),
])

# --------------------------------------------------------------------------------------------------
# Cell 7C2 score-blind final top-5 only. The score-bearing audit is deliberately not listed here.
# --------------------------------------------------------------------------------------------------
CELL_7C2_EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c2_quality_reranking_and_top5_v1'
)
CELL_7C2_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c2_quality_reranking_and_top5_v1'
)

CELL_7C2_SCORE_BLIND_TOP5 = (
    CELL_7C2_EXEC_DIR / 'cell_7c2_score_blind_final_top5_context_selection_v1.parquet'
)
EXPECTED_CELL_7C2_SCORE_BLIND_TOP5_SHA256 = (
    '88abb743262fe1dd2fede29f613893fd6a8a8a67c7d56efa75a961bf6d259c72'
)
CELL_7C2_MANIFEST = (
    CELL_7C2_CONFIG_DIR / 'cell_7c2_quality_reranking_and_top5_manifest_v1.json'
)
EXPECTED_CELL_7C2_MANIFEST_SHA256 = (
    '8a8e3dc827b4641f28922b8712c3024fb4c82ba1e3b4511213f28b9b4b1a83d5'
)

# --------------------------------------------------------------------------------------------------
# Exact Cell 7B3 score-blind prompt sources — resolved by exact filename + exact checksum.
# --------------------------------------------------------------------------------------------------
CELL_7B3_SCORE_BLIND = OrderedDict([
    ('semantic_corpus', {
        'filename': 'cell_7b3_score_blind_semantic_corpus_v1.parquet',
        'sha256': '2fead04f6c0814bb87207c9c36db7475370ae626f6a326c89ce672f402339399',
        'expected_rows': 100_920,
    }),
    ('primary_questions', {
        'filename': 'cell_7b3_primary_question_set_v1.csv',
        'sha256': 'c76e81952fcc6a698866b64da7b7daabeb281b7b9d10e17873596095b69d95df',
        'expected_rows': 80,
    }),
])

# --------------------------------------------------------------------------------------------------
# Exact Cell 7B4 frozen prompt configuration.
# --------------------------------------------------------------------------------------------------
CELL_7B4_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7b4_configuration_freeze_v1'
)
CELL_7B4_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7b4_configuration_freeze_v1'
)

CELL_7B4 = OrderedDict([
    ('llm_prompt_response', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_llm_prompt_response_configuration_v1.json',
        'sha256': 'e3f684f9c471b8074dc41f2398f8aff0cb03217f188e2eab2ad8f4c0070a810b',
    }),
    ('runtime_determinism', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_runtime_and_determinism_configuration_v1.json',
        'sha256': '6003c85ef151ae1d7dca530462fa1dbcf6983be4b7e92744b8e65c8d8b42b1d3',
    }),
    ('condition_aliases', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_condition_alias_inventory_v1.csv',
        'sha256': '6eb45683b42a456d2b6788a5fcf6b9cd95fc606afe9627610ebbc11914312cb9',
    }),
    ('manifest', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_configuration_freeze_manifest_v1.json',
        'sha256': '18a5d6cb3cadca0eab950839a19022686fc6bad2c398ed87f2a48c66eff462fe',
    }),
])

# --------------------------------------------------------------------------------------------------
# Cell 7C4 frozen outputs.
# --------------------------------------------------------------------------------------------------
EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c4_score_blind_prompt_materialization_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c4_score_blind_prompt_materialization_v1'
)
CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c4_score_blind_prompt_materialization_v1'
)

OUTPUTS = OrderedDict([
    ('prompt_instances',
     EXEC_DIR / 'cell_7c4_score_blind_prompt_instances_v1.parquet'),
    ('context_inventory',
     EXEC_DIR / 'cell_7c4_score_blind_prompt_context_inventory_v1.parquet'),
    ('input_inventory',
     CONFIG_DIR / 'cell_7c4_verified_prompt_input_inventory_v1.csv'),
    ('execution_report',
     QC_DIR / 'cell_7c4_prompt_materialization_execution_report_v1.json'),
    ('qc',
     QC_DIR / 'cell_7c4_prompt_materialization_qc_v1.json'),
    ('manifest',
     CONFIG_DIR / 'cell_7c4_prompt_materialization_manifest_v1.json'),
])

for directory in (EXEC_DIR, QC_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing_outputs = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing_outputs:
    raise FileExistsError(
        'Cell 7C4 fail-closed overwrite protection is active. Existing frozen output(s):\\n- '
        + '\\n- '.join(existing_outputs)
    )

print(f'Execution directory : {EXEC_DIR}')
print(f'QC directory        : {QC_DIR}')
print(f'Config directory    : {CONFIG_DIR}')

Execution directory : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/rag_execution/stage7_rag/cell_7c4_score_blind_prompt_materialization_v1
QC directory        : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c4_score_blind_prompt_materialization_v1
Config directory    : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c4_score_blind_prompt_materialization_v1


## 2. Strict checksum, sidecar, serialization, and deterministic-hash helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(
            f'Invalid SHA-256 sidecar format: {path}\\n'
            f'Observed first token: {token!r}'
        )
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    sc = sidecar_path(path)
    return (
        path.exists()
        and sc.exists()
        and read_sidecar_hash(sc) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing {label}: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'SHA-256 mismatch for {label}.\\n'
            f'Expected: {expected_sha256}\\nObserved: {observed}\\nPath: {path}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid or missing SHA-256 sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def locate_exact_hash(filename: str, expected_sha256: str) -> Path:
    candidates = sorted(path for path in ROOT.rglob(filename) if path.is_file())
    matches = [path for path in candidates if sha256_file(path) == expected_sha256]
    if len(matches) != 1:
        details = '\\n'.join(str(path) for path in candidates) or '<none>'
        raise RuntimeError(
            f'Expected exactly one checksum-matching artifact for {filename}; '
            f'found {len(matches)}.\\nCandidates:\\n{details}'
        )
    return matches[0]


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def parquet_metadata(path: Path) -> dict[str, Any]:
    pf = pq.ParquetFile(path)
    return {
        'rows': int(pf.metadata.num_rows),
        'columns': int(pf.metadata.num_columns),
        'schema_names': list(pf.schema_arrow.names),
    }


def csv_header_and_row_count(path: Path) -> tuple[list[str], int]:
    with path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.reader(handle)
        try:
            header = next(reader)
        except StopIteration:
            raise AssertionError(f'Empty CSV: {path}')
        rows = sum(1 for _ in reader)
    return header, rows


def to_json_native(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, np.generic):
        return to_json_native(value.item())
    if isinstance(value, np.ndarray):
        return [to_json_native(v) for v in value.tolist()]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): to_json_native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_json_native(v) for v in value]
    if hasattr(value, 'item'):
        return to_json_native(value.item())
    raise TypeError(f'Unsupported JSON type: {type(value).__name__}')


def stable_write_json(path: Path, payload: Any) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    native = to_json_native(payload)
    text = json.dumps(
        native,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    ) + '\n'
    path.write_text(text, encoding='utf-8')
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator='\n')
    return sha256_file(path)


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(
        path,
        index=False,
        engine='pyarrow',
        compression='zstd',
    )
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    digest = sha256_file(path)
    sidecar_path(path).write_text(f'{digest}  {path.name}\n', encoding='utf-8')


def canonical_message_bundle(system_prompt: str, user_prompt: str) -> str:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    return json.dumps(
        messages,
        ensure_ascii=False,
        sort_keys=True,
        separators=(',', ':'),
    )


# Fail early on the exact writer/sidecar convention.
with tempfile.TemporaryDirectory(prefix='cell_7c4_writer_selftest_') as _tmp:
    _tmpdir = Path(_tmp)

    _json_path = _tmpdir / 'test.json'
    _payload = {'ok': True, 'nested': {'value': 7}}
    stable_write_json(_json_path, _payload)

    _json_text = _json_path.read_text(encoding='utf-8')
    _json_bytes = _json_path.read_bytes()

    if json.loads(_json_text) != _payload:
        raise AssertionError('JSON round-trip self-test failed.')
    if not _json_bytes.endswith(b'\n'):
        raise AssertionError('JSON file does not end with a real LF newline byte.')
    if _json_bytes.endswith(b'\\n'):
        raise AssertionError('JSON writer emitted literal backslash+n instead of LF.')

    _csv_path = _tmpdir / 'test.csv'
    stable_write_csv(_csv_path, pd.DataFrame([{'a': 1}, {'a': 2}]))
    _csv_bytes = _csv_path.read_bytes()

    if b'\n' not in _csv_bytes:
        raise AssertionError('CSV writer did not emit real LF newlines.')
    if b'\\n' in _csv_bytes:
        raise AssertionError('CSV writer emitted literal backslash+n text.')

    write_sidecar(_json_path)
    _sidecar_bytes = sidecar_path(_json_path).read_bytes()

    if not sidecar_is_valid(_json_path):
        raise AssertionError('SHA-256 sidecar self-test failed.')
    if not _sidecar_bytes.endswith(b'\n'):
        raise AssertionError('SHA-256 sidecar does not end with a real LF newline byte.')

print('Serialization and SHA-256 helper self-test: PASS')


Serialization and SHA-256 helper self-test: PASS


## 3. Reverify the successful Cell 7C3 authorization package and exact downstream sources

In [4]:
verified_inputs: list[dict[str, Any]] = []

# Complete successful Cell 7C3 package.
verified_7c3 = OrderedDict()
for artifact_id, spec in CELL_7C3.items():
    record = verify_exact_artifact(
        f'cell_7c3_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C3'
    verified_7c3[artifact_id] = record
    verified_inputs.append(record)

auth_7c3 = load_json(CELL_7C3['authorization']['path'])
qc_7c3 = load_json(CELL_7C3['qc']['path'])
manifest_7c3 = load_json(CELL_7C3['manifest']['path'])

if auth_7c3.get('authorization_decision') != EXPECTED_CELL_7C3_AUTHORIZATION_DECISION:
    raise AssertionError('Cell 7C3 authorization decision mismatch.')
if auth_7c3.get('authorized_cell', {}).get('cell_id') != '7C4':
    raise AssertionError('Cell 7C3 does not authorize Cell 7C4.')
if auth_7c3.get('authorized_cell', {}).get('authorized_once') is not True:
    raise AssertionError('Cell 7C3 authorization must be single-use.')
if manifest_7c3.get('terminal_decision') != EXPECTED_CELL_7C3_TERMINAL_DECISION:
    raise AssertionError('Cell 7C3 terminal PASS decision mismatch.')
if manifest_7c3.get('next_authorized_cell') != '7C4':
    raise AssertionError('Cell 7C3 manifest does not authorize Cell 7C4.')
if manifest_7c3.get('llm_execution_authorized') is not False:
    raise AssertionError('Cell 7C3 must prohibit LLM execution.')
if int(qc_7c3.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C3 QC does not report zero failures.')

# Exact Cell 7C2 score-blind selection + manifest.
for label, path, expected_hash in [
    ('cell_7c2_score_blind_top5', CELL_7C2_SCORE_BLIND_TOP5, EXPECTED_CELL_7C2_SCORE_BLIND_TOP5_SHA256),
    ('cell_7c2_manifest', CELL_7C2_MANIFEST, EXPECTED_CELL_7C2_MANIFEST_SHA256),
]:
    record = verify_exact_artifact(label, path, expected_hash)
    record['source_cell'] = '7C2'
    verified_inputs.append(record)

manifest_7c2 = load_json(CELL_7C2_MANIFEST)
if manifest_7c2.get('terminal_decision') != EXPECTED_CELL_7C2_TERMINAL_DECISION:
    raise AssertionError('Cell 7C2 terminal PASS decision mismatch.')

top5_meta = parquet_metadata(CELL_7C2_SCORE_BLIND_TOP5)
EXPECTED_TOP5_SCHEMA = [
    'question_id',
    'blinded_alias',
    'context_position',
    'corpus_row_index',
    'packet_id',
    'rcv_accession',
]
if top5_meta['rows'] != EXPECTED_CONTEXT_ROWS:
    raise AssertionError(f'Score-blind top-5 row count changed: {top5_meta["rows"]}')
if top5_meta['schema_names'] != EXPECTED_TOP5_SCHEMA:
    raise AssertionError(
        'Score-blind top-5 schema changed.\\n'
        f'Observed: {top5_meta["schema_names"]}'
    )

# Exact Cell 7B3 score-blind prompt sources.
resolved_7b3 = {}
for key, spec in CELL_7B3_SCORE_BLIND.items():
    path = locate_exact_hash(spec['filename'], spec['sha256'])
    resolved_7b3[key] = path
    record = verify_exact_artifact(f'cell_7b3_{key}', path, spec['sha256'])
    record['source_cell'] = '7B3'
    verified_inputs.append(record)

semantic_meta = parquet_metadata(resolved_7b3['semantic_corpus'])
question_header, question_rows = csv_header_and_row_count(resolved_7b3['primary_questions'])

if semantic_meta['rows'] != 100_920:
    raise AssertionError(f'Semantic corpus row count changed: {semantic_meta["rows"]}')
if question_rows != EXPECTED_QUESTIONS:
    raise AssertionError(f'Primary question row count changed: {question_rows}')

required_semantic_columns = {
    'evidence_packet_id',
    'rcv_accession',
    'semantic_evidence_text',
}
if not required_semantic_columns.issubset(set(semantic_meta['schema_names'])):
    raise AssertionError(
        'Semantic corpus is missing required score-blind fields.\\n'
        f'Observed: {semantic_meta["schema_names"]}'
    )

if not {'question_id', 'question_text'}.issubset(set(question_header)):
    raise AssertionError(
        'Primary-question file is missing question_id/question_text.\\n'
        f'Observed: {question_header}'
    )

# Exact Cell 7B4 frozen prompt config.
verified_7b4 = OrderedDict()
for artifact_id, spec in CELL_7B4.items():
    record = verify_exact_artifact(
        f'cell_7b4_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7B4'
    verified_7b4[artifact_id] = record
    verified_inputs.append(record)

manifest_7b4 = load_json(CELL_7B4['manifest']['path'])
if manifest_7b4.get('terminal_decision') != EXPECTED_CELL_7B4_TERMINAL_DECISION:
    raise AssertionError('Cell 7B4 terminal decision mismatch.')

print('Cell 7C3 authorization package             : 4/4 VERIFIED')
print('Cell 7C3 terminal PASS                     : VERIFIED')
print('Cell 7C4 authorization                     : VERIFIED')
print('LLM execution authorization                : FALSE')
print(f'Score-blind top-5                          : {top5_meta["rows"]:,} rows')
print(f'Score-blind semantic corpus                : {semantic_meta["rows"]:,} rows')
print(f'Primary questions                          : {question_rows}')
print('Score-bearing Cell 7C2 audit loaded        : NO')
print('Cell 7A3 score table loaded                 : NO')
print('Answer-key outcomes loaded                 : NO')

Cell 7C3 authorization package             : 4/4 VERIFIED
Cell 7C3 terminal PASS                     : VERIFIED
Cell 7C4 authorization                     : VERIFIED
LLM execution authorization                : FALSE
Score-blind top-5                          : 2,400 rows
Score-blind semantic corpus                : 100,920 rows
Primary questions                          : 80
Score-bearing Cell 7C2 audit loaded        : NO
Cell 7A3 score table loaded                 : NO
Answer-key outcomes loaded                 : NO


## 4. Validate the exact frozen prompt templates, aliases, model snapshot, and future generation settings

In [5]:
llm_config = load_json(CELL_7B4['llm_prompt_response']['path'])
runtime_config = load_json(CELL_7B4['runtime_determinism']['path'])

llm_cfg = llm_config.get('llm', {})
generation_cfg = llm_config.get('generation', {})
prompt_cfg = llm_config.get('prompt', {})
structured_cfg = llm_config.get('structured_output', {})
response_schema = structured_cfg.get('schema', {})

with CELL_7B4['condition_aliases']['path'].open('r', encoding='utf-8', newline='') as handle:
    alias_rows = list(csv.DictReader(handle))

if len(alias_rows) != EXPECTED_ALIASES:
    raise AssertionError(f'Expected six frozen blinded aliases; observed {len(alias_rows)}.')

expected_alias_order = ['ARM-MICA', 'ARM-ORBIT', 'ARM-KITE', 'ARM-PULSE', 'ARM-LARCH', 'ARM-NOVA']
observed_alias_order = [row['blinded_alias'] for row in alias_rows]
if observed_alias_order != expected_alias_order:
    raise AssertionError(
        'Frozen blinded alias order changed.\\n'
        f'Observed: {observed_alias_order}'
    )

expected_required_response_fields = {
    'answer',
    'clinical_significance',
    'conflict_detected',
    'evidence_strength',
    'response_policy',
    'confidence',
    'evidence_ids',
    'reasoning_summary',
}

prompt_config_checks = OrderedDict([
    ('llm_provider_openai', llm_cfg.get('provider') == 'OpenAI'),
    ('llm_api_responses', llm_cfg.get('api') == 'Responses API'),
    ('llm_snapshot_exact', llm_cfg.get('model') == 'gpt-4.1-mini-2025-04-14'),
    ('fixed_snapshot_required', llm_cfg.get('fixed_snapshot_required') is True),
    ('tools_disabled', llm_cfg.get('tools') == []),
    ('web_search_disabled', llm_cfg.get('web_search') is False),
    ('file_search_disabled', llm_cfg.get('file_search') is False),
    ('code_interpreter_disabled', llm_cfg.get('code_interpreter') is False),
    ('temperature_zero', generation_cfg.get('temperature') == 0.0),
    ('top_p_one', generation_cfg.get('top_p') == 1.0),
    ('max_output_tokens_1200', generation_cfg.get('max_output_tokens') == 1200),
    ('repetitions_three', generation_cfg.get('repetitions_per_question_condition') == 3),
    ('run_ids_exact', generation_cfg.get('run_ids') == [0, 1, 2]),
    ('question_position_exact', prompt_cfg.get('question_position') == 'before evidence context'),
    ('context_order_exact', prompt_cfg.get('context_order') == 'frozen final top-5 order'),
    ('score_values_in_prompt_false', prompt_cfg.get('score_values_in_prompt') is False),
    ('condition_identity_in_prompt_false', prompt_cfg.get('condition_identity_in_prompt') is False),
    ('answer_key_in_prompt_false', prompt_cfg.get('answer_key_in_prompt') is False),
    ('strict_json_schema', structured_cfg.get('strict') is True),
    ('response_schema_type_exact', structured_cfg.get('type') == 'json_schema'),
    ('response_schema_required_exact', set(response_schema.get('required', [])) == expected_required_response_fields),
    ('response_schema_properties_exact', set(response_schema.get('properties', {}).keys()) == expected_required_response_fields),
    ('response_schema_additional_properties_false', response_schema.get('additionalProperties') is False),
])

failed_prompt_config_checks = [
    name for name, passed in prompt_config_checks.items() if not bool(passed)
]
if failed_prompt_config_checks:
    raise RuntimeError(
        'Frozen Cell 7B4 prompt configuration validation failed:\\n- '
        + '\\n- '.join(failed_prompt_config_checks)
    )

SYSTEM_PROMPT = prompt_cfg.get('system_prompt')
USER_PROMPT_TEMPLATE = prompt_cfg.get('user_prompt_template')
CONTEXT_BLOCK_TEMPLATE = prompt_cfg.get('context_block_template')

for name, value in [
    ('system_prompt', SYSTEM_PROMPT),
    ('user_prompt_template', USER_PROMPT_TEMPLATE),
    ('context_block_template', CONTEXT_BLOCK_TEMPLATE),
]:
    if not isinstance(value, str) or not value.strip():
        raise AssertionError(f'Missing frozen prompt text: {name}')

# Exact frozen text hashes must reproduce.
for text_key, hash_key in [
    ('system_prompt', 'system_prompt_sha256'),
    ('user_prompt_template', 'user_prompt_template_sha256'),
    ('context_block_template', 'context_block_template_sha256'),
]:
    observed = sha256_text(prompt_cfg[text_key])
    expected = prompt_cfg.get(hash_key)
    if observed != expected:
        raise AssertionError(
            f'Frozen prompt text hash mismatch for {text_key}.\\n'
            f'Expected: {expected}\\nObserved: {observed}'
        )

required_user_prompt_placeholders = {
    '{question_id}',
    '{question_text}',
    '{context_blocks}',
}
if not all(token in USER_PROMPT_TEMPLATE for token in required_user_prompt_placeholders):
    raise AssertionError(
        'Frozen user prompt template is missing one or more required placeholders.'
    )

required_context_placeholders = {
    '{context_position}',
    '{packet_id}',
    '{rcv_accession}',
    '{semantic_text}',
}
if not all(token in CONTEXT_BLOCK_TEMPLATE for token in required_context_placeholders):
    raise AssertionError(
        'Frozen context-block template is missing one or more required placeholders.'
    )

print('Frozen prompt templates                     : VERIFIED')
print('User-template placeholders                  : question_id, question_text, context_blocks')
print('Context-template placeholders               : context_position, packet_id, rcv_accession, semantic_text')
print(f'System prompt SHA-256                       : {prompt_cfg["system_prompt_sha256"]}')
print(f'User template SHA-256                       : {prompt_cfg["user_prompt_template_sha256"]}')
print(f'Context template SHA-256                    : {prompt_cfg["context_block_template_sha256"]}')
print('Frozen LLM snapshot                         : gpt-4.1-mini-2025-04-14')
print('Future repetitions                          : 3')
print('Future total calls if separately authorized  : 1,440')
print('LLM called in Cell 7C4                      : NO')

Frozen prompt templates                     : VERIFIED
User-template placeholders                  : question_id, question_text, context_blocks
Context-template placeholders               : context_position, packet_id, rcv_accession, semantic_text
System prompt SHA-256                       : 68a78ec6d78f0f3e11f88d4b941bf2806a7ffde8b35073887c6d3a8003f39ec6
User template SHA-256                       : adebf4dba3128c5d0fde875c207466cf5c8d4a6aa889d72bb6f9e04abd3747c1
Context template SHA-256                    : 08f876c82ff47a9472a3fdab26642c293964724675c17e4a8e0ecc005c7fba2e
Frozen LLM snapshot                         : gpt-4.1-mini-2025-04-14
Future repetitions                          : 3
Future total calls if separately authorized  : 1,440
LLM called in Cell 7C4                      : NO


## 5. Load only authorized score-blind rows and materialize 2,400 context blocks + 480 prompts

In [6]:
# Snapshot exact upstream bytes before authorized row-level reads.
IMMUTABLE_SOURCE_PATHS = OrderedDict([
    ('cell_7c3_authorization', CELL_7C3['authorization']['path']),
    ('cell_7c3_manifest', CELL_7C3['manifest']['path']),
    ('cell_7c2_score_blind_top5', CELL_7C2_SCORE_BLIND_TOP5),
    ('cell_7c2_manifest', CELL_7C2_MANIFEST),
    ('cell_7b3_semantic_corpus', resolved_7b3['semantic_corpus']),
    ('cell_7b3_primary_questions', resolved_7b3['primary_questions']),
    ('cell_7b4_prompt_config', CELL_7B4['llm_prompt_response']['path']),
    ('cell_7b4_runtime_config', CELL_7B4['runtime_determinism']['path']),
    ('cell_7b4_aliases', CELL_7B4['condition_aliases']['path']),
    ('cell_7b4_manifest', CELL_7B4['manifest']['path']),
])
immutable_hashes_before = OrderedDict(
    (key, sha256_file(path)) for key, path in IMMUTABLE_SOURCE_PATHS.items()
)

top5 = pd.read_parquet(CELL_7C2_SCORE_BLIND_TOP5)
semantic = pd.read_parquet(resolved_7b3['semantic_corpus'])
questions = pd.read_csv(
    resolved_7b3['primary_questions'],
    dtype={'question_id': 'string', 'question_text': 'string'},
)

# Canonical string dtypes.
for column in ('question_id', 'blinded_alias', 'packet_id', 'rcv_accession'):
    top5[column] = top5[column].astype('string')
top5['context_position'] = top5['context_position'].astype('int16')
top5['corpus_row_index'] = top5['corpus_row_index'].astype('int64')

semantic = semantic.rename(columns={
    'evidence_packet_id': 'packet_id',
    'semantic_evidence_text': 'evidence_text',
}).copy()
semantic['packet_id'] = semantic['packet_id'].astype('string')
semantic['rcv_accession'] = semantic['rcv_accession'].astype('string')
semantic['evidence_text'] = semantic['evidence_text'].astype('string')

questions['question_id'] = questions['question_id'].astype('string')
questions['question_text'] = questions['question_text'].astype('string')

# Input integrity.
if len(top5) != EXPECTED_CONTEXT_ROWS:
    raise AssertionError(f'Expected 2,400 score-blind selections; observed {len(top5):,}.')
if top5['question_id'].nunique(dropna=False) != EXPECTED_QUESTIONS:
    raise AssertionError('Score-blind selection does not contain exactly 80 questions.')
if top5['blinded_alias'].nunique(dropna=False) != EXPECTED_ALIASES:
    raise AssertionError('Score-blind selection does not contain exactly six aliases.')
if set(top5['blinded_alias'].astype(str)) != set(expected_alias_order):
    raise AssertionError('Score-blind selection contains an unexpected blinded alias.')
if top5[['question_id', 'blinded_alias', 'context_position']].duplicated().any():
    raise AssertionError('Duplicate question/alias/context_position row found.')
if not top5.groupby(['question_id', 'blinded_alias']).size().eq(5).all():
    raise AssertionError('Every question/alias must have exactly five selected context rows.')
if not top5.groupby(['question_id', 'blinded_alias'])['context_position'].apply(
    lambda s: tuple(sorted(int(v) for v in s.tolist())) == (1, 2, 3, 4, 5)
).all():
    raise AssertionError('Every question/alias must contain context positions exactly 1 through 5.')

if len(semantic) != 100_920:
    raise AssertionError(f'Expected 100,920 semantic rows; observed {len(semantic):,}.')
if semantic['packet_id'].isna().any() or semantic['packet_id'].duplicated().any():
    raise AssertionError('Semantic corpus packet_id is missing or duplicated.')
if semantic['rcv_accession'].isna().any():
    raise AssertionError('Semantic corpus contains missing RCV accession.')
if semantic['evidence_text'].isna().any() or semantic['evidence_text'].str.strip().eq('').any():
    raise AssertionError('Semantic corpus contains missing/blank evidence text.')

if len(questions) != EXPECTED_QUESTIONS:
    raise AssertionError(f'Expected 80 questions; observed {len(questions):,}.')
if questions['question_id'].isna().any() or questions['question_id'].duplicated().any():
    raise AssertionError('Primary-question IDs are missing or duplicated.')
if questions['question_text'].isna().any() or questions['question_text'].str.strip().eq('').any():
    raise AssertionError('Primary-question text contains missing/blank values.')

# Join selected packets to exactly one score-blind semantic row.
semantic_subset = semantic[['packet_id', 'rcv_accession', 'evidence_text']].copy()
context_join = top5.merge(
    semantic_subset,
    on='packet_id',
    how='left',
    validate='many_to_one',
    indicator=True,
    suffixes=('_selected', '_semantic'),
    sort=False,
)

if not context_join['_merge'].eq('both').all():
    missing = int(context_join['_merge'].ne('both').sum())
    raise AssertionError(f'{missing} selected packets failed to join to the semantic corpus.')
context_join = context_join.drop(columns=['_merge'])

if not context_join['rcv_accession_selected'].eq(context_join['rcv_accession_semantic']).all():
    mismatches = int(
        context_join['rcv_accession_selected'].ne(context_join['rcv_accession_semantic']).sum()
    )
    raise AssertionError(f'{mismatches} packet joins changed the frozen RCV identity.')

context_join = context_join.rename(
    columns={'rcv_accession_selected': 'rcv_accession'}
).drop(columns=['rcv_accession_semantic'])

# Join questions one-to-many.
context_join = context_join.merge(
    questions[['question_id', 'question_text']],
    on='question_id',
    how='left',
    validate='many_to_one',
    indicator=True,
    sort=False,
)
if not context_join['_merge'].eq('both').all():
    raise AssertionError('At least one selected context row failed to join to a primary question.')
context_join = context_join.drop(columns=['_merge'])

alias_order_map = {alias: i for i, alias in enumerate(expected_alias_order)}
context_join['_alias_order'] = context_join['blinded_alias'].map(alias_order_map).astype('int16')
context_join = context_join.sort_values(
    ['question_id', '_alias_order', 'context_position'],
    kind='mergesort',
).reset_index(drop=True)

# Render the exact frozen context block.
context_join['evidence_text_sha256'] = context_join['evidence_text'].map(sha256_text)
context_join['context_block'] = [
    CONTEXT_BLOCK_TEMPLATE.format(
        context_position=int(position),
        packet_id=str(packet_id),
        rcv_accession=str(rcv),
        semantic_text=str(evidence_text),
    )
    for position, packet_id, rcv, evidence_text in zip(
        context_join['context_position'],
        context_join['packet_id'],
        context_join['rcv_accession'],
        context_join['evidence_text'],
    )
]
context_join['context_block_sha256'] = context_join['context_block'].map(sha256_text)

# Final score-blind context inventory. No quality/semantic/RRF/condition identity fields.
context_inventory = context_join[
    [
        'question_id',
        'blinded_alias',
        'context_position',
        'corpus_row_index',
        'packet_id',
        'rcv_accession',
        'evidence_text_sha256',
        'context_block',
        'context_block_sha256',
    ]
].copy()

prompt_rows: list[dict[str, Any]] = []

for (question_id, blinded_alias), group in context_join.groupby(
    ['question_id', 'blinded_alias'],
    sort=False,
):
    group = group.sort_values('context_position', kind='mergesort')

    if tuple(group['context_position'].astype(int).tolist()) != (1, 2, 3, 4, 5):
        raise AssertionError(
            f'Frozen context order changed for {question_id} / {blinded_alias}.'
        )

    question_text_values = group['question_text'].astype(str).unique().tolist()
    if len(question_text_values) != 1:
        raise AssertionError(
            f'Question text is not unique within {question_id} / {blinded_alias}.'
        )
    question_text = question_text_values[0]

    context_blocks = '\\n\\n'.join(group['context_block'].astype(str).tolist())
    user_prompt = USER_PROMPT_TEMPLATE.format(
        question_id=str(question_id),
        question_text=question_text,
        context_blocks=context_blocks,
    )
    message_bundle = canonical_message_bundle(SYSTEM_PROMPT, user_prompt)

    packet_ids = group['packet_id'].astype(str).tolist()
    rcvs = group['rcv_accession'].astype(str).tolist()
    context_block_hashes = group['context_block_sha256'].astype(str).tolist()

    prompt_rows.append({
        'prompt_instance_id': f'{question_id}__{blinded_alias}',
        'question_id': str(question_id),
        'blinded_alias': str(blinded_alias),
        'question_text_sha256': sha256_text(question_text),
        'context_count': 5,
        'context_packet_ids_json': json.dumps(packet_ids, ensure_ascii=False, separators=(',', ':')),
        'context_rcv_accessions_json': json.dumps(rcvs, ensure_ascii=False, separators=(',', ':')),
        'context_block_hashes_json': json.dumps(context_block_hashes, ensure_ascii=False, separators=(',', ':')),
        'context_bundle_sha256': sha256_text(context_blocks),
        'system_prompt': SYSTEM_PROMPT,
        'system_prompt_sha256': sha256_text(SYSTEM_PROMPT),
        'user_prompt': user_prompt,
        'user_prompt_sha256': sha256_text(user_prompt),
        'full_prompt_sha256': sha256_text(message_bundle),
    })

prompt_instances = pd.DataFrame(prompt_rows)

prompt_instances['_alias_order'] = prompt_instances['blinded_alias'].map(alias_order_map).astype('int16')
prompt_instances = prompt_instances.sort_values(
    ['question_id', '_alias_order'],
    kind='mergesort',
).drop(columns=['_alias_order']).reset_index(drop=True)

context_inventory = context_inventory.copy()
context_inventory['_alias_order'] = context_inventory['blinded_alias'].map(alias_order_map).astype('int16')
context_inventory = context_inventory.sort_values(
    ['question_id', '_alias_order', 'context_position'],
    kind='mergesort',
).drop(columns=['_alias_order']).reset_index(drop=True)

print(f'Selected context rows loaded              : {len(top5):,}')
print(f'Semantic context joins                    : {len(context_join):,}/{len(top5):,}')
print(f'Score-blind context rows materialized      : {len(context_inventory):,}')
print(f'Score-blind prompt instances materialized : {len(prompt_instances):,}')
print('Prompt contents displayed                 : NO')
print('Score-bearing reranking audit loaded       : NO')
print('Cell 7A3 score table loaded                : NO')
print('Answer keys loaded                         : NO')
print('LLM called                                 : NO')

Selected context rows loaded              : 2,400
Semantic context joins                    : 2,400/2,400
Score-blind context rows materialized      : 2,400
Score-blind prompt instances materialized : 480
Prompt contents displayed                 : NO
Score-bearing reranking audit loaded       : NO
Cell 7A3 score table loaded                : NO
Answer keys loaded                         : NO
LLM called                                 : NO


## 6. Fail-closed prompt-materialization QC before writing any frozen artifact

In [7]:
prewrite_checks = OrderedDict()

# Counts and key structure.
prewrite_checks['prompt_rows_480'] = len(prompt_instances) == EXPECTED_PROMPTS
prewrite_checks['context_rows_2400'] = len(context_inventory) == EXPECTED_CONTEXT_ROWS
prewrite_checks['prompt_80_questions'] = prompt_instances['question_id'].nunique() == EXPECTED_QUESTIONS
prewrite_checks['prompt_6_aliases'] = prompt_instances['blinded_alias'].nunique() == EXPECTED_ALIASES
prewrite_checks['one_prompt_per_question_alias'] = (
    not prompt_instances.duplicated(['question_id', 'blinded_alias']).any()
)
prewrite_checks['five_contexts_per_prompt'] = prompt_instances['context_count'].eq(5).all()
prewrite_checks['context_five_rows_per_question_alias'] = (
    context_inventory.groupby(['question_id', 'blinded_alias']).size().eq(5).all()
)
prewrite_checks['context_positions_exact_1_to_5'] = (
    context_inventory.groupby(['question_id', 'blinded_alias'])['context_position']
    .apply(lambda s: tuple(s.astype(int).tolist()) == (1, 2, 3, 4, 5))
    .all()
)

# Exact source membership and RCV identity.
top5_identity = (
    top5[
        ['question_id', 'blinded_alias', 'context_position', 'corpus_row_index', 'packet_id', 'rcv_accession']
    ]
    .sort_values(
        ['question_id', 'blinded_alias', 'context_position'],
        kind='mergesort',
    )
    .reset_index(drop=True)
)
context_identity = (
    context_inventory[
        ['question_id', 'blinded_alias', 'context_position', 'corpus_row_index', 'packet_id', 'rcv_accession']
    ]
    .sort_values(
        ['question_id', 'blinded_alias', 'context_position'],
        kind='mergesort',
    )
    .reset_index(drop=True)
)
prewrite_checks['exact_cell7c2_top5_identity_preserved'] = context_identity.equals(top5_identity)

# Reproduce every rendered context block from frozen inputs.
reproduced_context_blocks = [
    CONTEXT_BLOCK_TEMPLATE.format(
        context_position=int(row.context_position),
        packet_id=str(row.packet_id),
        rcv_accession=str(row.rcv_accession),
        semantic_text=str(
            context_join.loc[
                (context_join['question_id'].eq(row.question_id))
                & (context_join['blinded_alias'].eq(row.blinded_alias))
                & (context_join['context_position'].eq(row.context_position)),
                'evidence_text',
            ].iloc[0]
        ),
    )
    for row in context_inventory.itertuples(index=False)
]
prewrite_checks['all_context_blocks_exact_template_reproduction'] = (
    context_inventory['context_block'].astype(str).tolist() == reproduced_context_blocks
)
prewrite_checks['all_context_block_hashes_exact'] = all(
    sha256_text(block) == digest
    for block, digest in zip(
        context_inventory['context_block'].astype(str),
        context_inventory['context_block_sha256'].astype(str),
    )
)

# Reproduce every prompt from exact question + frozen contexts.
prompt_reproduction_ok = True
for row in prompt_instances.itertuples(index=False):
    group = context_inventory.loc[
        context_inventory['question_id'].eq(row.question_id)
        & context_inventory['blinded_alias'].eq(row.blinded_alias)
    ].sort_values('context_position', kind='mergesort')

    question_text = questions.loc[
        questions['question_id'].eq(row.question_id),
        'question_text',
    ]
    if len(question_text) != 1:
        prompt_reproduction_ok = False
        break

    context_blocks = '\\n\\n'.join(group['context_block'].astype(str).tolist())
    expected_user_prompt = USER_PROMPT_TEMPLATE.format(
        question_id=str(row.question_id),
        question_text=str(question_text.iloc[0]),
        context_blocks=context_blocks,
    )

    if expected_user_prompt != row.user_prompt:
        prompt_reproduction_ok = False
        break
    if sha256_text(expected_user_prompt) != row.user_prompt_sha256:
        prompt_reproduction_ok = False
        break
    if sha256_text(SYSTEM_PROMPT) != row.system_prompt_sha256:
        prompt_reproduction_ok = False
        break
    if sha256_text(context_blocks) != row.context_bundle_sha256:
        prompt_reproduction_ok = False
        break
    if sha256_text(canonical_message_bundle(SYSTEM_PROMPT, expected_user_prompt)) != row.full_prompt_sha256:
        prompt_reproduction_ok = False
        break

prewrite_checks['all_480_prompts_exact_template_reproduction'] = prompt_reproduction_ok

# No unblinded condition identity or score-bearing columns.
PROHIBITED_COLUMN_TOKENS = (
    'condition_id',
    'condition_name',
    'condition_role',
    'ges',
    'p_stable',
    'instability',
    'quality_signal',
    'quality_rank',
    'semantic_rank',
    'semantic_score',
    'rrf',
    'random_quality',
)
leaking_prompt_columns = [
    column
    for column in prompt_instances.columns
    if any(token in column.lower() for token in PROHIBITED_COLUMN_TOKENS)
]
leaking_context_columns = [
    column
    for column in context_inventory.columns
    if any(token in column.lower() for token in PROHIBITED_COLUMN_TOKENS)
]
prewrite_checks['prompt_artifact_has_no_score_or_condition_identity_columns'] = (
    len(leaking_prompt_columns) == 0
)
prewrite_checks['context_artifact_has_no_score_or_condition_identity_columns'] = (
    len(leaking_context_columns) == 0
)

# Prompt metadata must preserve blinded aliases only.
prewrite_checks['prompt_aliases_exact'] = set(prompt_instances['blinded_alias']) == set(expected_alias_order)
prewrite_checks['context_aliases_exact'] = set(context_inventory['blinded_alias']) == set(expected_alias_order)
prewrite_checks['no_unblinded_condition_columns'] = (
    'condition_id' not in prompt_instances.columns
    and 'condition_name' not in prompt_instances.columns
    and 'condition_role' not in prompt_instances.columns
)

# Frozen prompt configuration is used exactly.
prewrite_checks['system_prompt_hash_exact'] = prompt_instances['system_prompt_sha256'].eq(
    prompt_cfg['system_prompt_sha256']
).all()
prewrite_checks['system_prompt_text_exact'] = prompt_instances['system_prompt'].eq(SYSTEM_PROMPT).all()
prewrite_checks['question_position_before_context'] = (
    prompt_cfg.get('question_position') == 'before evidence context'
)
prewrite_checks['context_order_frozen_top5'] = (
    prompt_cfg.get('context_order') == 'frozen final top-5 order'
)
prewrite_checks['score_values_in_prompt_false'] = prompt_cfg.get('score_values_in_prompt') is False
prewrite_checks['condition_identity_in_prompt_false'] = prompt_cfg.get('condition_identity_in_prompt') is False
prewrite_checks['answer_key_in_prompt_false'] = prompt_cfg.get('answer_key_in_prompt') is False

# Cell 7C4 scientific boundary.
prewrite_checks['score_bearing_audit_not_loaded'] = True
prewrite_checks['cell7a3_scores_not_loaded'] = True
prewrite_checks['answer_keys_not_loaded'] = True
prewrite_checks['llm_not_called'] = True
prewrite_checks['responses_not_generated'] = True
prewrite_checks['adjudication_not_performed'] = True
prewrite_checks['rag_metrics_not_calculated'] = True

failed_prewrite_checks = [
    name for name, passed in prewrite_checks.items()
    if not bool(passed)
]
if failed_prewrite_checks:
    raise RuntimeError(
        'Cell 7C4 failed before writing any frozen output. Failed checks:\\n- '
        + '\\n- '.join(failed_prewrite_checks)
    )

print(f'Prewrite QC checks                        : {len(prewrite_checks)}/{len(prewrite_checks)} PASS')
print('Exact Cell 7C2 top-5 identity             : PRESERVED')
print('Exact frozen context-block template       : VERIFIED')
print('Exact frozen user-prompt template         : VERIFIED')
print('Prompt score/condition leakage boundary   : VERIFIED')
print('LLM called                                : NO')
print('Frozen Cell 7C4 artifacts written so far  : NO')

Prewrite QC checks                        : 31/31 PASS
Exact Cell 7C2 top-5 identity             : PRESERVED
Exact frozen context-block template       : VERIFIED
Exact frozen user-prompt template         : VERIFIED
Prompt score/condition leakage boundary   : VERIFIED
LLM called                                : NO
Frozen Cell 7C4 artifacts written so far  : NO


## 7. Freeze prompt/context package, QC, execution report, manifest, and SHA-256 sidecars

In [8]:
input_inventory = pd.DataFrame([
    {
        'input_id': record['input_id'],
        'source_cell': record['source_cell'],
        'path': record['path'],
        'sha256': record['sha256'],
        'bytes': record['bytes'],
        'sidecar_path': record['sidecar_path'],
        'sidecar_valid': record['sidecar_valid'],
        'row_level_content_loaded_in_cell_7c4': (
            record['input_id'] in {
                'cell_7c2_score_blind_top5',
                'cell_7b3_semantic_corpus',
                'cell_7b3_primary_questions',
                'cell_7b4_llm_prompt_response',
                'cell_7b4_condition_aliases',
            }
        ),
    }
    for record in verified_inputs
])

terminal_decision = (
    'PASS_STAGE7C4_480_SCORE_BLIND_PROMPTS_AND_2400_CONTEXT_BLOCKS_MATERIALIZED_'
    'FROM_EXACT_CELL7C2_TOP5_CELL7B3_SCORE_BLIND_CORPUS_AND_QUESTIONS_USING_EXACT_'
    'CELL7B4_FROZEN_TEMPLATES_CHECKSUM_PROTECTED_NO_SCORE_BEARING_AUDIT_CELL7A3_SCORES_'
    'GES_QUALITY_RANK_SEMANTIC_RANK_RRF_UNBLINDED_CONDITION_IDENTITY_ANSWER_KEYS_'
    'LLM_RESPONSES_ADJUDICATION_OR_RAG_METRICS_NEXT_EXECUTION_NOT_AUTHORIZED'
)

with tempfile.TemporaryDirectory(prefix='cell_7c4_staging_') as tmpdir_text:
    tmpdir = Path(tmpdir_text)
    staged = {key: tmpdir / path.name for key, path in OUTPUTS.items()}

    # Scientific artifacts.
    stable_write_parquet(staged['prompt_instances'], prompt_instances)
    stable_write_parquet(staged['context_inventory'], context_inventory)
    stable_write_csv(staged['input_inventory'], input_inventory)

    staged_hashes = OrderedDict(
        (key, sha256_file(staged[key]))
        for key in ('prompt_instances', 'context_inventory', 'input_inventory')
    )

    execution_report = {
        'cell_id': CELL_ID,
        'stage': STAGE,
        'package_version': PACKAGE_VERSION,
        'created_utc': CREATED_UTC,
        'notebook': NOTEBOOK_NAME,
        'project_root': str(ROOT),
        'authorization': {
            'cell_7c3_authorization_path': str(CELL_7C3['authorization']['path']),
            'cell_7c3_authorization_sha256': CELL_7C3['authorization']['sha256'],
            'cell_7c3_manifest_sha256': CELL_7C3['manifest']['sha256'],
            'authorization_decision': EXPECTED_CELL_7C3_AUTHORIZATION_DECISION,
            'authorized_cell': '7C4',
        },
        'frozen_prompt_design': {
            'questions': EXPECTED_QUESTIONS,
            'blinded_aliases': expected_alias_order,
            'prompt_instances': EXPECTED_PROMPTS,
            'context_blocks_per_prompt': EXPECTED_CONTEXTS_PER_PROMPT,
            'context_rows': EXPECTED_CONTEXT_ROWS,
            'question_position': prompt_cfg['question_position'],
            'context_order': prompt_cfg['context_order'],
            'system_prompt_sha256': prompt_cfg['system_prompt_sha256'],
            'user_prompt_template_sha256': prompt_cfg['user_prompt_template_sha256'],
            'context_block_template_sha256': prompt_cfg['context_block_template_sha256'],
            'score_values_in_prompt': False,
            'condition_identity_in_prompt': False,
            'answer_key_in_prompt': False,
        },
        'future_generation_frozen_but_not_authorized': {
            'model_snapshot': llm_cfg['model'],
            'api': llm_cfg['api'],
            'temperature': generation_cfg['temperature'],
            'top_p': generation_cfg['top_p'],
            'max_output_tokens': generation_cfg['max_output_tokens'],
            'repetitions_per_question_condition': generation_cfg['repetitions_per_question_condition'],
            'run_ids': generation_cfg['run_ids'],
            'expected_total_calls': EXPECTED_FUTURE_CALLS,
            'llm_execution_authorized': False,
        },
        'execution_accounting': {
            'score_blind_top5_rows_loaded': int(len(top5)),
            'semantic_corpus_rows_loaded': int(len(semantic)),
            'primary_questions_loaded': int(len(questions)),
            'semantic_context_joins': int(len(context_join)),
            'context_rows_materialized': int(len(context_inventory)),
            'prompt_instances_materialized': int(len(prompt_instances)),
        },
        'scientific_operations': {
            'score_bearing_cell7c2_audit_loaded': False,
            'cell7a3_scores_loaded': False,
            'prompts_materialized': True,
            'llm_called': False,
            'responses_generated': False,
            'answer_keys_loaded_or_inspected': False,
            'adjudication_performed': False,
            'rag_or_retrieval_metrics_calculated': False,
        },
        'output_artifacts': {
            key: {'path': str(OUTPUTS[key]), 'sha256': staged_hashes[key]}
            for key in ('prompt_instances', 'context_inventory', 'input_inventory')
        },
        'terminal_decision': terminal_decision,
        'next_authorized_cell': None,
        'next_required_action': (
            'Create a separate fail-closed authorization before any OpenAI Responses API call, '
            'model response generation, answer-key inspection, adjudication, or RAG evaluation.'
        ),
    }
    stable_write_json(staged['execution_report'], execution_report)
    staged_hashes['execution_report'] = sha256_file(staged['execution_report'])

    qc_payload = {
        'cell_id': CELL_ID,
        'stage': STAGE,
        'package_version': PACKAGE_VERSION,
        'created_utc': CREATED_UTC,
        'prompt_config_checks': {
            name: bool(value) for name, value in prompt_config_checks.items()
        },
        'prewrite_checks': {
            name: bool(value) for name, value in prewrite_checks.items()
        },
        'passed_checks': int(len(prompt_config_checks) + len(prewrite_checks)),
        'failed_checks': 0,
        'total_checks': int(len(prompt_config_checks) + len(prewrite_checks)),
        'prompt_rows': int(len(prompt_instances)),
        'context_rows': int(len(context_inventory)),
        'leaking_prompt_columns': leaking_prompt_columns,
        'leaking_context_columns': leaking_context_columns,
        'terminal_decision': terminal_decision,
    }
    stable_write_json(staged['qc'], qc_payload)
    staged_hashes['qc'] = sha256_file(staged['qc'])

    manifest_payload = {
        'cell_id': CELL_ID,
        'stage': STAGE,
        'package_version': PACKAGE_VERSION,
        'created_utc': CREATED_UTC,
        'notebook': NOTEBOOK_NAME,
        'project_root': str(ROOT),
        'authorization_lineage': {
            'cell_7c3_authorization_sha256': CELL_7C3['authorization']['sha256'],
            'cell_7c3_manifest_sha256': CELL_7C3['manifest']['sha256'],
            'authorization_decision': EXPECTED_CELL_7C3_AUTHORIZATION_DECISION,
        },
        'upstream_lineage': {
            'cell_7c2_score_blind_top5_sha256': EXPECTED_CELL_7C2_SCORE_BLIND_TOP5_SHA256,
            'cell_7c2_manifest_sha256': EXPECTED_CELL_7C2_MANIFEST_SHA256,
            'cell_7b3_semantic_corpus_sha256': CELL_7B3_SCORE_BLIND['semantic_corpus']['sha256'],
            'cell_7b3_primary_questions_sha256': CELL_7B3_SCORE_BLIND['primary_questions']['sha256'],
            'cell_7b4_prompt_response_config_sha256': CELL_7B4['llm_prompt_response']['sha256'],
            'cell_7b4_runtime_config_sha256': CELL_7B4['runtime_determinism']['sha256'],
            'cell_7b4_aliases_sha256': CELL_7B4['condition_aliases']['sha256'],
            'cell_7b4_manifest_sha256': CELL_7B4['manifest']['sha256'],
        },
        'frozen_output_artifacts': {
            key: {'path': str(OUTPUTS[key]), 'sha256': staged_hashes[key]}
            for key in (
                'prompt_instances',
                'context_inventory',
                'input_inventory',
                'execution_report',
                'qc',
            )
        },
        'scientific_boundary': {
            'prompt_instances_materialized': True,
            'context_rows_materialized': True,
            'score_bearing_cell7c2_audit_loaded': False,
            'cell7a3_scores_loaded': False,
            'llm_called': False,
            'responses_generated': False,
            'answer_keys_loaded_or_inspected': False,
            'adjudication_performed': False,
            'rag_or_retrieval_metrics_calculated': False,
        },
        'terminal_decision': terminal_decision,
        'next_authorized_cell': None,
        'llm_execution_authorized': False,
        'next_required_action': (
            'Separate fail-closed LLM-generation authorization is required.'
        ),
    }
    stable_write_json(staged['manifest'], manifest_payload)

    # Complete staged readback before copying anything to frozen Drive locations.
    for key, path in staged.items():
        if not path.exists() or path.stat().st_size == 0:
            raise AssertionError(f'Staged Cell 7C4 artifact missing/empty: {key}')

    staged_prompts = pd.read_parquet(staged['prompt_instances'])
    staged_contexts = pd.read_parquet(staged['context_inventory'])
    staged_input_inventory = pd.read_csv(staged['input_inventory'])
    staged_report = load_json(staged['execution_report'])
    staged_qc = load_json(staged['qc'])
    staged_manifest = load_json(staged['manifest'])

    staged_checks = OrderedDict([
        ('staged_prompt_rows_480', len(staged_prompts) == EXPECTED_PROMPTS),
        ('staged_context_rows_2400', len(staged_contexts) == EXPECTED_CONTEXT_ROWS),
        ('staged_context_5_per_question_alias',
         staged_contexts.groupby(['question_id', 'blinded_alias']).size().eq(5).all()),
        ('staged_prompt_unique_question_alias',
         not staged_prompts.duplicated(['question_id', 'blinded_alias']).any()),
        ('staged_input_inventory_nonempty', len(staged_input_inventory) > 0),
        ('staged_report_llm_false',
         staged_report['scientific_operations']['llm_called'] is False),
        ('staged_qc_zero_failures', int(staged_qc['failed_checks']) == 0),
        ('staged_manifest_terminal_decision_exact',
         staged_manifest['terminal_decision'] == terminal_decision),
        ('staged_manifest_next_cell_none',
         staged_manifest['next_authorized_cell'] is None),
        ('staged_manifest_llm_false',
         staged_manifest['llm_execution_authorized'] is False),
    ])
    failed_staged_checks = [
        name for name, passed in staged_checks.items() if not bool(passed)
    ]
    if failed_staged_checks:
        raise RuntimeError(
            'Staged Cell 7C4 package readback failed; nothing copied to final paths:\\n- '
            + '\\n- '.join(failed_staged_checks)
        )

    # Commit fully verified staged package.
    for key, final_path in OUTPUTS.items():
        final_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(staged[key], final_path)
        write_sidecar(final_path)

# Fresh final readback from Drive.
for key, path in OUTPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing final Cell 7C4 output: {path}')
    if not sidecar_is_valid(path):
        raise AssertionError(f'Final Cell 7C4 sidecar failed: {path}')

readback_prompts = pd.read_parquet(OUTPUTS['prompt_instances'])
readback_contexts = pd.read_parquet(OUTPUTS['context_inventory'])
readback_report = load_json(OUTPUTS['execution_report'])
readback_qc = load_json(OUTPUTS['qc'])
readback_manifest = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('final_prompt_rows_480', len(readback_prompts) == EXPECTED_PROMPTS),
    ('final_context_rows_2400', len(readback_contexts) == EXPECTED_CONTEXT_ROWS),
    ('final_prompts_80_questions', readback_prompts['question_id'].nunique() == EXPECTED_QUESTIONS),
    ('final_prompts_6_aliases', readback_prompts['blinded_alias'].nunique() == EXPECTED_ALIASES),
    ('final_unique_question_alias',
     not readback_prompts.duplicated(['question_id', 'blinded_alias']).any()),
    ('final_context_5_per_question_alias',
     readback_contexts.groupby(['question_id', 'blinded_alias']).size().eq(5).all()),
    ('final_context_positions_1_to_5',
     readback_contexts.groupby(['question_id', 'blinded_alias'])['context_position']
     .apply(lambda s: tuple(s.astype(int).tolist()) == (1, 2, 3, 4, 5)).all()),
    ('final_report_llm_false', readback_report['scientific_operations']['llm_called'] is False),
    ('final_qc_zero_failures', int(readback_qc['failed_checks']) == 0),
    ('final_manifest_decision_exact', readback_manifest['terminal_decision'] == terminal_decision),
    ('final_manifest_next_cell_none', readback_manifest['next_authorized_cell'] is None),
    ('final_manifest_llm_false', readback_manifest['llm_execution_authorized'] is False),
    ('all_six_output_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_readback_checks = [
    name for name, passed in readback_checks.items() if not bool(passed)
]
if failed_readback_checks:
    raise RuntimeError(
        'Final Cell 7C4 frozen-package readback failed:\\n- '
        + '\\n- '.join(failed_readback_checks)
    )

# Reconfirm every upstream artifact remained byte-identical.
immutable_hashes_after = OrderedDict(
    (key, sha256_file(path)) for key, path in IMMUTABLE_SOURCE_PATHS.items()
)
if immutable_hashes_after != immutable_hashes_before:
    changed = [
        key for key in immutable_hashes_before
        if immutable_hashes_before[key] != immutable_hashes_after[key]
    ]
    raise AssertionError(
        'One or more frozen upstream artifacts changed during Cell 7C4:\\n- '
        + '\\n- '.join(changed)
    )

total_checks = (
    len(prompt_config_checks)
    + len(prewrite_checks)
    + len(staged_checks)
    + len(readback_checks)
)
passed_checks = total_checks

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C4')
print('SCORE-BLIND PROMPT MATERIALIZATION')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM AUTHORIZATION AND SOURCE REVERIFICATION')
print(f'Cell 7C3 manifest SHA-256                     : {sha256_file(CELL_7C3["manifest"]["path"])}')
print('Cell 7C3 terminal PASS verified               : YES')
print('Cell 7C4 prompt-materialization authorization : VERIFIED')
print(f'Cell 7C2 score-blind top-5 SHA-256            : {sha256_file(CELL_7C2_SCORE_BLIND_TOP5)}')
print(f'Cell 7B3 semantic corpus SHA-256               : {sha256_file(resolved_7b3["semantic_corpus"])}')
print(f'Cell 7B3 primary questions SHA-256             : {sha256_file(resolved_7b3["primary_questions"])}')
print(f'Cell 7B4 prompt config SHA-256                 : {sha256_file(CELL_7B4["llm_prompt_response"]["path"])}')
print('Score-bearing Cell 7C2 audit loaded           : NO')
print('Cell 7A3 scores loaded                        : NO')
print('Answer keys loaded                            : NO')

print('\\nSCORE-BLIND PROMPT MATERIALIZATION')
print(f'Primary questions                             : {EXPECTED_QUESTIONS}')
print(f'Blinded aliases                               : {EXPECTED_ALIASES}')
print(f'Prompt instances                              : {len(readback_prompts):,}')
print(f'Context blocks                                : {len(readback_contexts):,}')
print('Context blocks per prompt                     : 5')
print('Question position                             : before evidence context')
print('Context order                                 : frozen context_position 1 through 5')
print('Score values in prompt                        : NO')
print('A-F condition identity in prompt              : NO')
print('Answer key in prompt                          : NO')
print('Prompt contents displayed                     : NO')

print('\\nFROZEN FUTURE GENERATION SETTINGS — NOT YET AUTHORIZED')
print(f'Model snapshot                                : {llm_cfg["model"]}')
print(f'Temperature                                   : {generation_cfg["temperature"]}')
print(f'Top-p                                         : {generation_cfg["top_p"]}')
print(f'Max output tokens                             : {generation_cfg["max_output_tokens"]}')
print(f'Planned repetitions                           : {EXPECTED_FUTURE_REPETITIONS}')
print(f'Planned calls                                 : {EXPECTED_FUTURE_CALLS:,}')
print('LLM execution                                 : NOT AUTHORIZED')

print('\\nCELL 7C4 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {passed_checks}/{total_checks} PASS')

print('\\nSCIENTIFIC OPERATIONS IN CELL 7C4')
print('Score-blind prompts materialized              : YES')
print('Score-blind context inventory materialized    : YES')
print('LLM called                                   : NO')
print('Responses generated                          : NO')
print('Answer-key outcomes inspected                : NO')
print('Adjudication or RAG metrics                  : NO')

print('\\nNEXT AUTHORIZATION BOUNDARY')
print('Next execution cell                           : NOT AUTHORIZED')
print('Required next action                          : separate fail-closed authorization before')
print('                                                 1,440 planned LLM response calls')
print('Prompt package allowed downstream             : Cell 7C4 score-blind prompts ONLY')
print('Score-bearing reranking audit downstream      : PROHIBITED')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C4
SCORE-BLIND PROMPT MATERIALIZATION
Notebook                                      : 11_GES_Aware_Genomic_RAG_Cell_7C4_Score_Blind_Prompt_Materialization_V3.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM AUTHORIZATION AND SOURCE REVERIFICATION
Cell 7C3 manifest SHA-256                     : fc2ddb9593baf025e688b12254757fa9125e667ea57ad19104b2179cb5243f0c
Cell 7C3 terminal PASS verified               : YES
Cell 7C4 prompt-materialization authorization : VERIFIED
Cell 7C2 score-blind top-5 SHA-256            : 88abb743262fe1dd2fede29f613893fd6a8a8a67c7d56efa75a961bf6d259c72
Cell 7B3 semantic corpus SHA-256               : 2fead04f6c0814bb87207c9c36db7475370ae626f6a326c89ce672f402339399
Cell 7B3 primary questions SHA-256             : 